
# Set up a PyChemkin project

This example shows how to set up a PyChemkin project and start to use
PyChemkin features. You begin by importing the PyChemkin package,
which is named ``ansys.chemkin.core``, and optionally the PyChemkin
logger package, which is named ``ansys.chemkin.core.logger``.

Next, you use the ``Chemistry()`` method to instantiate and
preprocess a chemistry set. This requires specifying the file names
(with full file paths) of the mechanism file, thermodynamic data file,
and optional transport data file.

Once the chemistry set instance is instantiated, you use the
``preprocess()`` method to preprocess the mechanism data. You can then
start to use PyChemkin features to build your simulation.

<div class="alert alert-info"><h4>Note</h4><p>Running PyChemkin requires Ansys 2025 R2 or later.</p></div>


## Import PyChemkin packages
Import the ``ansys.chemkin.core`` package to start using PyChemkin in the project.
Importing the ``ansys.chemkin.core.logger`` package is optional.



In [ ]:
from pathlib import Path

import ansys.chemkin.core  # import PyChemkin
from ansys.chemkin.core.logger import logger

## Set up the file paths of the mechanism and data files
The ``ansys_dir`` variable on your local computer specifies
the location of your Ansys installation.

Use ``os.path`` methods to construct the file names with the full paths.

Assign the files to the corresponding chemistry set arguments:

- ``mech_file``: Mechanism input file
- ``therm_file``: Thermodynamic data file
- ``tran_file``: Transport data file (optional)




In [ ]:
# create GRI 3.0 mechanism from the data directory
mechanism_dir = Path(ansys.chemkin.core.ansys_dir) / "reaction" / "data"
# set up mechanism file names
mech_file = str(mechanism_dir / "grimech30_chem.inp")
therm_file = str(mechanism_dir / "grimech30_thermo.dat")
tran_file = str(mechanism_dir / "grimech30_transport.dat")

## Instantiate the chemistry set
Use the ``Chemistry()`` method to instantiate a chemistry set named ``GasMech``.



In [ ]:
GasMech = ansys.chemkin.core.Chemistry(
    chem=mech_file, therm=therm_file, tran=tran_file, label="GRI 3.0"
)

## Preprocess the chemistry set
Preprocess the GRI 3.0 chemistry set.



In [ ]:
status = GasMech.preprocess()

## Check the preprocessing status



In [ ]:
if status != 0:
    # failed
    print(f"Preprocessing: Error encountered. Code = {status:d}.")
    print(f"See the summary file {GasMech.summaryfile} for details.")
    logger.error("Preprocessing failed.")
    exit()

## Start to use PyChemkin features
Start to use PyChemkin features to build your simulation.
For example, using the ``Mixture()`` method, create an ``air`` mixture
based on the ``GasMech`` chemistry set.



In [ ]:
# create 'air' mixture based on 'GasMech' chemistry set
air = ansys.chemkin.core.Mixture(GasMech)
# set 'air' condition
# mixture pressure in [dynes/cm2]
air.pressure = 1.0 * ansys.chemkin.core.P_ATM
# mixture temperature in [K]
air.temperature = 300.0
# mixture composition in mole fractions
air.x = [("O2", 0.21), ("N2", 0.79)]

## Print the properties of the air mixture for verification

<div class="alert alert-info"><h4>Note</h4><p>- The default units of temperature and pressure are [K] and
    [dynes/cm\ :sup:`2`\ ], respectively.
  - The constant ``P_ATM`` is a conversion multiplier for pressure.
  - Transport property methods such as ``mixture_viscosity()`` require
    transport data. You must include the transport data file when
    creating the chemistry set.</p></div>




In [ ]:
# print pressure and temperature of the `air` mixture
print(f"Pressure    = {air.pressure / ansys.chemkin.core.P_ATM} [atm]")
print(f"Temperature = {air.temperature} [K]")
# print the 'air' composition in mass fractions
air.list_composition(mode="mass")
# get 'air' mixture density [g/cm3]
print(f"Mixture density   = {air.rho} [g/cm3]")
# get 'air' mixture viscosity [g/cm-sec] or [poise]
print(f"Mixture viscosity = {air.mixture_viscosity() * 100.0} [cP]")

# clean up
ansys.chemkin.core.done()